# Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
# Loading the raw data file
data_path = os.path.join("..", "data", "resume_extraction.csv")
df = pd.read_csv(data_path, on_bad_lines="skip", encoding="utf-8", encoding_errors="replace", low_memory=False)
df.columns = df.columns.str.strip()

print("Data loaded successfully.")

Data loaded successfully.


In [ ]:
# Possible placeholder values in the data
placeholder = ["No", "NA", "NO", ""]

# Iterating through column names and replacing placeholder values with NaN
placeholder_column = ["Education", "Specialization", "interests", "skills", "Job_title", "Highest Qualification"]
for col in placeholder_column:
    if col in df.columns:
        df[col] = df[col].replace(placeholder, np.nan)
        df[col] = df[col].replace("", np.nan)

salary_col = "Yearly salary in pounds"
df[salary_col] = pd.to_numeric(df[salary_col], errors="coerce")

# Map of possible majors
education_map = {
    "b.e": "BE", "be": "BE", "b.tech": "B.Tech", "btech": "B.Tech",
    "b.sc": "B.Sc", "bsc": "B.Sc", "bca": "BCA", "b.com": "B.Com", "bcom": "B.Com",
    "ba": "BA", "mba": "MBA", "m.sc": "M.Sc", "msc": "M.Sc", "m.tech": "M.Tech", "mtech": "M.Tech",
    "m.com": "M.Com", "mcom": "M.Com", "bms": "BMS", "b.m.s": "BMS",
    "b.pharmacy": "B.Pharmacy", "bpharmacy": "B.Pharmacy", "diploma": "Diploma",
    "mca": "MCA", "b.allb": "BALLB", "ballb": "BALLB", "law hons": "Law Hons",
    "bachelor of management studies": "BMS", "bachelor of journalism": "Bachelor of Journalism",
}

# Making values lowercase and replacing with values from the map
df["Education"] = df["Education"].astype(str).str.strip().str.lower()
df["Education"] = df["Education"].replace(education_map)

# Making specialization column lowercase and replacing values
df["Specialization"] = df["Specialization"].astype(str).str.strip().str.lower()
df["Specialization"] = df["Specialization"].replace(["nan", ""], np.nan)

df["Gender"] = df["Gender"].astype(str).str.strip()
df["Gender"] = df["Gender"].replace(["nan", ""], np.nan)

print("Preprocessing complete.")

Preprocessing complete.


In [ ]:
def format(val):
    if pd.isna(val) or (isinstance(val, str) and val.strip() in ["", "NO", "No", "NA"]):
        return []
    s = str(val).strip()
    
    # Replace semicolons
    s = s.replace(";", ",")
    parts = [p.strip() for p in s.split(",") if p.strip()]
    return list(dict.fromkeys(parts))  # deduplicating

df["skills_list"] = df["skills"].apply(format)
df["interests_list"] = df["interests"].apply(format)

print("Sample parsed skills:", df["skills_list"].iloc[0][:5], "...")
print("Sample parsed interests:", df["interests_list"].iloc[2][:3], "...")

Sample parsed skills: ['Critical Thinking', 'Analytic Thinking', 'SQL', 'Programming', 'Work under Pressure'] ...
Sample parsed interests: ['Sales/Marketing', 'Trading', 'Understand human behaviour'] ...


In [ ]:
certifications = "Certifications"

# Replacing null values and converting to binary
df["has_certification"] = ((df[certifications].notna()) & (~df[certifications].astype(str).str.strip().str.lower().isin(["no", "nan", ""]))).astype(int)

df["is_employed"] = (df["Job_status"].astype(str).str.strip().str.lower() == "yes").astype(int)

# Encoding education level
education_level_map = {"Diploma": 1, "BCA": 2, "BA": 2, "B.Com": 2, "B.Sc": 2, "BE": 2, "B.Tech": 2, "BMS": 2, "B.Pharmacy": 2, "BALLB": 2, "Law Hons": 2, "Bachelor of Journalism": 2, "MBA": 3, "M.Sc": 3, "M.Tech": 3, "M.Com": 3, "MCA": 3,}

df["education_level"] = df["Education"].map(education_level_map).fillna(2)

# Keywords per domain
stem_words = ["computer", "engineering", "mechanical", "electrical", "electronics", "civil", "technology", "math", "physics", "chemistry", "biology", "biotech", "ai", "data"]
business_words = ["commerce", "marketing", "management", "account", "finance", "mba", "sales"]
humanities_words = ["english", "psychology", "history", "political", "law", "journalism"]

def domain_set(spec):
    if pd.isna(spec) or spec == "nan": return "Other"
    s = str(spec).lower()
    if any(k in s for k in stem_words): return "STEM"
    if any(k in s for k in business_words): return "Business"
    if any(k in s for k in humanities_words): return "Humanities"

    return "Other"

df["specialization_domain"] = df["Specialization"].apply(domain_set)

df["skill_count"] = df["skills_list"].apply(len)
df["interest_count"] = df["interests_list"].apply(len)

# Setting keywords per category
programming = ["python", "java", "sql", "c++", "javascript", "r", "c", "php", "matlab", "cpp"]
soft_skills = ["communication", "leadership", "critical thinking", "problem solving", "team work", "teamwork"]

def check_skill(skills_list, keywords):
    skills_lower = [s.lower() for s in skills_list]
    return any(any(kw in s for kw in keywords) for s in skills_lower)

def count_words(skills_list, keywords):
    return sum(1 for s in skills_list if any(kw in s.lower() for kw in keywords))

df["has_programming_skills"] = df["skills_list"].apply(lambda x: check_skill(x, programming)).astype(int)
df["has_soft_skills"] = df["skills_list"].apply(lambda x: check_skill(x, soft_skills)).astype(int)

df["tech_skill_count"] = df["skills_list"].apply(lambda x: count_words(x, programming))
df["soft_skill_count"] = df["skills_list"].apply(lambda x: count_words(x, soft_skills))

# Setting tech keywords
tech_words = ["software", "developer", "engineer", "data", "analyst", "programmer", "it", "tech"]

def match_job_education(row):
    domain = str(row["specialization_domain"]).lower()
    title = str(row["Job_title"]).lower() if pd.notna(row["Job_title"]) else ""

    if domain == "stem" and any(k in title for k in tech_words): return 1
    if domain == "business" and any(k in title for k in ["manager", "sales", "finance", "marketing"]): return 1
    
    return 0

df["education_job_match"] = df.apply(match_job_education, axis=1)

# highest education level encoding
level_encoding = {"bachelor": 1, "bachelors": 1, "b.sc": 1, "b.tech": 1, "be": 1,"master": 2, "masters": 2, "m.sc": 2, "m.tech": 2, "mba": 2, "mca": 2, "phd": 3, "doctorate": 3}

def qualification(val):
    if pd.isna(val): return 1
    s = str(val).lower()

    for a, b in level_encoding.items():
        if a in s: return b
    return 1

df["highest_qualification_level"] = df["Highest Qualification"].apply(qualification)

# Creating salary features and encoding into buckets
df["yearly_salary"] = df["Yearly salary in pounds"]
salary_buckets = pd.cut(df["yearly_salary"], bins=[0, 50, 70, 200], labels=["low", "medium", "high"])
df["salary_bucket"] = salary_buckets.astype(str).replace("nan", "unknown")

# Adjusting job title and fetching length
df["job_title_length"] = df["Job_title"].fillna("").astype(str).str.len()
df["job_title_clean"] = df["Job_title"].fillna("").astype(str).str.strip().str.lower()

print("Feature creation complete.")

Feature creation complete.


In [ ]:
# Feature columns for model
features = [
    "education_level", "specialization_domain", "has_certification", "is_employed",
    "skill_count", "interest_count", "has_programming_skills", "has_soft_skills",
    "tech_skill_count", "soft_skill_count", "education_job_match", "highest_qualification_level",
    "yearly_salary", "salary_bucket", "job_title_length"
]

# Building feature df for model excluding gender
df_features = df[features + ["Gender"]].copy()

# filling categorical features missing values
df_features["salary_bucket"] = df_features["salary_bucket"].fillna("unknown")
df_features["specialization_domain"] = df_features["specialization_domain"].fillna("Other")

salary_median = df_features["yearly_salary"].median()
df_features["yearly_salary"] = df_features["yearly_salary"].fillna(salary_median)

print("Final feature matrix shape:", df_features.shape)
print("Columns:", list(df_features.columns))
print("\nGender distribution (for bias analysis):")
print(df_features["Gender"].value_counts(dropna=False))

Final feature matrix shape: (1048575, 16)
Columns: ['education_level', 'specialization_domain', 'has_certification', 'is_employed', 'skill_count', 'interest_count', 'has_programming_skills', 'has_soft_skills', 'tech_skill_count', 'soft_skill_count', 'education_job_match', 'highest_qualification_level', 'yearly_salary', 'salary_bucket', 'job_title_length', 'Gender']

Gender distribution (for bias analysis):
Gender
Male                 583717
NaN                  244226
Female               220398
Prefer not to say       234
Name: count, dtype: int64


In [ ]:
write_feature = os.path.join("..", "data", "resume_features.csv")
df_features.to_csv(write_feature, index=False)
print(f"Saved to {write_feature}")

print("\n--- Validation ---")
print("Duplicate rows:", df_features.duplicated().sum())
print("Row count:", len(df_features))

print("\n--- Categorical feature value counts ---")
for col in ["specialization_domain", "salary_bucket"]:
    print(f"\n{col}:")
    print(df_features[col].value_counts(dropna=False).head(10))

print("\n--- Numeric feature summary ---")
numeric_cols = ["education_level", "skill_count", "interest_count", "yearly_salary", "job_title_length"]
print(df_features[numeric_cols].describe())

Saved to ..\data\resume_features.csv

--- Validation ---
Duplicate rows: 998544
Row count: 1048575

--- Categorical feature value counts ---

specialization_domain:
specialization_domain
STEM          581548
Other         407437
Business       33446
Humanities     26144
Name: count, dtype: int64

salary_bucket:
salary_bucket
medium    511570
low       281247
high      255758
Name: count, dtype: int64

--- Numeric feature summary ---
       education_level   skill_count  interest_count  yearly_salary  \
count     1.048575e+06  1.048575e+06    1.048575e+06   1.048575e+06   
mean      2.000176e+00  4.866855e+00    2.400565e+00   5.999866e+01   
std       2.106866e-01  4.501140e+00    2.824577e+00   1.183180e+01   
min       1.000000e+00  0.000000e+00    0.000000e+00   4.000000e+01   
25%       2.000000e+00  1.000000e+00    1.000000e+00   5.000000e+01   
50%       2.000000e+00  5.000000e+00    2.000000e+00   6.000000e+01   
75%       2.000000e+00  7.000000e+00    3.000000e+00   7.000000e+0